# Solar Filament Instance Segmentation — Pipeline Walkthrough

This notebook demonstrates the end-to-end preprocessing, leak-free validation split, full-disk inference, and Panoptic Quality (PQ) evaluation used for the **MAGFiLO Solar Filament Segmentation** challenge.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, str(Path("..").resolve()))

from filament_seg.dataset import load_annotations, make_folds
from filament_seg.inference import predict_full_disk
from filament_seg.metrics import evaluate_pq, mask_from_array
from filament_seg.model import load_model
from filament_seg.postprocess import erode_masks, extract_instances
from filament_seg.preprocess import prepare_channels
from filament_seg.rle import mask_to_rle, rle_to_mask

## 1. Load Annotations and Build Observation-Level Folds

About 42% of the 707 training observations are labeled by two or three solar physicists under distinct `image_id` keys. Grouping splits strictly by `file_name` prevents the same physical H-alpha frame from appearing in both train and validation sets.

In [ ]:
DATA_DIR = Path("../data/MAGFiLO_1.0_Kaggle_2026")
ANN_PATH = DATA_DIR / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
IMG_DIR = DATA_DIR / "train" / "train_images"

gt_groups, meta = load_annotations(ANN_PATH)
folds = make_folds(meta, n_splits=5, seed=42)

print(f"Unique observations: {meta['file_name'].nunique()}")
print(f"Annotation units:    {len(gt_groups)}")
print("Fold distribution:\n", folds.value_counts().sort_index())

## 2. Full-Disk H-Alpha Channel Preparation

Each `2048x2048` grayscale observation is mapped to a 3-channel `[0, 1]` tensor: percentile-clipped disk intensity, local contrast (CLAHE), and normalized Sobel gradient magnitude.

In [ ]:
sample_file = sorted(folds[folds == 0].index)[0]
gray = cv2.imread(str(IMG_DIR / sample_file), cv2.IMREAD_GRAYSCALE)
channels = prepare_channels(gray)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, ch, title in zip(axes, channels, ["Disk Normalized", "CLAHE Contrast", "Sobel Gradient"]):
    ax.imshow(ch, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## 3. Single-Pass 2048x2048 Inference and Boundary Calibration

Although models are trained on `640x640` crops, running inference on the uncropped `2048x2048` frame avoids tile-seam fragmentation and improves validation PQ while cutting runtime per frame.

In [ ]:
WEIGHTS_PATH = Path("../weights/fold0_best.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"

if WEIGHTS_PATH.exists():
    model = load_model(str(WEIGHTS_PATH), device=device)
    probs = predict_full_disk(model, channels, tile=2048, use_tta=True, device=device)
    instances = erode_masks(extract_instances(probs), iterations=1)

    # Round-trip RLE sanity check
    if instances:
        rle = mask_to_rle(instances[0])
        decoded = rle_to_mask(rle)
        assert np.array_equal(instances[0].astype(bool), decoded)

    pred_masks = [m for m in (mask_from_array(inst) for inst in instances) if m is not None]
    sample_gt = {k: v for k, v in gt_groups.items() if k[0] == sample_file}
    res = evaluate_pq(sample_gt, {sample_file: pred_masks})
    print(res.as_dict())